# E3 — LSTM và GRU trên chuỗi ảnh

Ảnh Fashion-MNIST được xem như chuỗi hàng hoặc chuỗi patch. Bốn mô hình recurrent được so sánh với MLP/CNN E1 trên đúng split.


## 1. Môi trường


### 1.1. Cài đặt thư viện

**Mục tiêu:** Kiểm tra các dependency nền tảng và chỉ cài những gói còn thiếu trong runtime.

**Kết quả:** Môi trường sẵn sàng cho xử lý Fashion-MNIST, huấn luyện recurrent model và xuất biểu đồ.


In [ ]:
import importlib.util, subprocess, sys

required = {
    "torch": "torch>=2.2,<3",
    "torchvision": "torchvision>=0.17,<1",
    "numpy": "numpy>=1.26,<3",
    "pandas": "pandas>=2.1,<4",
    "matplotlib": "matplotlib>=3.8,<4",
    "seaborn": "seaborn>=0.13,<1",
    "sklearn": "scikit-learn>=1.4,<2",
    "tqdm": "tqdm>=4.66,<5",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencies ready")


### 1.2. Import và chọn thiết bị

**Mục tiêu:** Nạp thư viện dùng chung, chọn CPU/GPU và bật mixed precision khi có CUDA.

**Kết quả:** Tạo `DEVICE`, `AMP_ENABLED` và các namespace dùng trong notebook.


In [ ]:
import copy
import csv
import hashlib
import json
import math
import os
import platform
import random
import time
from contextlib import nullcontext
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"
print("device:", DEVICE)


## 2. Cấu hình


### 2.1. Khai báo cấu hình chạy

**Mục tiêu:** Thiết lập đường dẫn, seed, batch size, epoch, resume, chế độ `smoke`/`full` và experiment ID.

**Kết quả:** Tạo dictionary `CONFIG` điều khiển toàn bộ pipeline E3.


In [ ]:
def default_root(kind):
    env_name = "CO5085_DATA_ROOT" if kind == "data" else "CO5085_OUTPUT_ROOT"
    if os.getenv(env_name):
        return Path(os.environ[env_name])
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working") / ("data" if kind == "data" else "artifacts")
    if Path("/content").exists():
        return Path("/content") / ("data" if kind == "data" else "artifacts")
    return Path("./data" if kind == "data" else "./artifacts")


CONFIG = {
    "data_root": str(default_root("data")),
    "output_root": str(default_root("output")),
    "run_mode": os.getenv("CO5085_RUN_MODE", "smoke"),
    "experiment_ids": [],  # Empty means all experiments for this notebook.
    "seed": 42,
    "batch_size": 128,
    "epochs": 20,
    "resume_from": None,  # None, "last", or a checkpoint path.
    "evaluate_only": os.getenv("CO5085_EVALUATE_ONLY", "0") == "1",
    "num_workers": 2,
}
assert CONFIG["run_mode"] in {"smoke", "full"}
if CONFIG["run_mode"] == "smoke":
    CONFIG["epochs"] = 1
    CONFIG["num_workers"] = 0
print(json.dumps(CONFIG, indent=2))


### 2.2. Khởi tạo tiện ích thực nghiệm

**Mục tiêu:** Định nghĩa các hàm seed, split hash, checkpoint, training loop, metric, benchmark và trực quan hóa.

**Kết quả:** Cung cấp hạ tầng dùng chung cho baseline và recurrent model.


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def environment_info():
    return {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "sklearn": sklearn.__version__,
        "cuda": torch.version.cuda,
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    }


def json_dump(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, ensure_ascii=False, default=str)


def atomic_torch_save(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(value, temporary)
    temporary.replace(path)


def rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def restore_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and state.get("cuda") is not None:
        torch.cuda.set_rng_state_all(state["cuda"])


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 2**20
    return {"parameters": total, "trainable_parameters": trainable, "weight_size_mb": size_mb}


def split_digest(train_idx, val_idx):
    digest = hashlib.sha256()
    digest.update(np.asarray(train_idx, dtype=np.int64).tobytes())
    digest.update(np.asarray(val_idx, dtype=np.int64).tobytes())
    return digest.hexdigest()


def fashion_split(dataset, data_root, seed=42):
    split_path = Path(data_root) / "fashion_mnist_split_seed42.npz"
    labels = np.asarray(dataset.targets)
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=6000, random_state=seed)
    expected_train, expected_val = next(splitter.split(np.zeros(len(labels)), labels))
    if split_path.exists():
        saved = np.load(split_path)
        train_idx, val_idx = saved["train_idx"], saved["val_idx"]
        if not (np.array_equal(train_idx, expected_train) and np.array_equal(val_idx, expected_val)):
            raise RuntimeError(f"Split file does not match seed={seed}: {split_path}")
    else:
        split_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez(split_path, train_idx=expected_train, val_idx=expected_val)
        train_idx, val_idx = expected_train, expected_val
    assert len(train_idx) == 54000 and len(val_idx) == 6000
    assert not set(train_idx).intersection(set(val_idx))
    return train_idx, val_idx, split_digest(train_idx, val_idx)


def classification_epoch(model, loader, criterion, optimizer=None, scaler=None,
                         max_batches=None, grad_clip=None, grad_accum=1, frozen_eval=None):
    training = optimizer is not None
    model.train(training)
    if training and frozen_eval is not None:
        frozen_eval(model)
    if training:
        optimizer.zero_grad(set_to_none=True)
    total_loss = total_correct = total_items = 0
    accumulation_count = 0
    start = time.perf_counter()
    context = torch.enable_grad if training else torch.no_grad
    with context():
        for batch_index, (images, labels) in enumerate(loader):
            if max_batches is not None and batch_index >= max_batches:
                break
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
                logits = model(images)
                loss = criterion(logits, labels)
                backward_loss = loss / grad_accum
            if training:
                scaler.scale(backward_loss).backward()
                accumulation_count += 1
                should_step = accumulation_count == grad_accum
                if should_step:
                    if grad_clip is not None:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    accumulation_count = 0
            total_loss += loss.item() * labels.size(0)
            total_correct += (logits.argmax(1) == labels).sum().item()
            total_items += labels.size(0)
        if training and accumulation_count:
            if grad_clip is not None:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
    elapsed = time.perf_counter() - start
    return {"loss": total_loss / total_items, "accuracy": total_correct / total_items,
            "items": total_items, "seconds": elapsed}


@torch.no_grad()
def predict_classifier(model, loader, max_batches=None):
    model.eval()
    labels_all, predictions_all, probabilities_all = [], [], []
    for batch_index, (images, labels) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        logits = model(images.to(DEVICE))
        probabilities = logits.softmax(1).cpu()
        labels_all.append(labels.cpu())
        predictions_all.append(probabilities.argmax(1))
        probabilities_all.append(probabilities)
    return (torch.cat(labels_all).numpy(), torch.cat(predictions_all).numpy(),
            torch.cat(probabilities_all).numpy())


def resolve_resume(run_dir, resume_from):
    if resume_from is None:
        return None
    if resume_from == "last":
        candidate = Path(run_dir) / "last.pt"
    else:
        candidate = Path(resume_from)
    return candidate if candidate.exists() else None


def fit_classifier(model, loaders, optimizer, scheduler, run_dir, run_config,
                   grad_clip=None, grad_accum=1, frozen_eval=None):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    json_dump(run_config, run_dir / "config.json")
    json_dump(environment_info(), run_dir / "environment.json")
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    start_epoch, best_accuracy, best_loss, history = 0, -1.0, float("inf"), []
    if run_config.get("evaluate_only"):
        evaluation_path = run_dir / "best.pt"
        if not evaluation_path.exists():
            raise FileNotFoundError(f"evaluate_only requires {evaluation_path}")
        checkpoint = torch.load(evaluation_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint["model"])
        start_epoch = run_config["epochs"]
        best_accuracy, best_loss = checkpoint["best_accuracy"], checkpoint["best_loss"]
        history_path = run_dir / "history.csv"
        if history_path.exists():
            history = pd.read_csv(history_path).to_dict("records")
    else:
        resume_path = resolve_resume(run_dir, run_config.get("resume_from"))
        checkpoint = torch.load(resume_path, map_location=DEVICE, weights_only=False) if resume_path is not None else None
    if not run_config.get("evaluate_only") and checkpoint is not None:
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        if scheduler is not None and checkpoint.get("scheduler") is not None:
            scheduler.load_state_dict(checkpoint["scheduler"])
        if checkpoint.get("scaler") is not None:
            scaler.load_state_dict(checkpoint["scaler"])
        restore_rng_state(checkpoint["rng_state"])
        start_epoch = checkpoint["epoch"] + 1
        best_accuracy, best_loss = checkpoint["best_accuracy"], checkpoint["best_loss"]
        history_path = run_dir / "history.csv"
        if history_path.exists():
            history = pd.read_csv(history_path).to_dict("records")

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    for epoch in range(start_epoch, run_config["epochs"]):
        train_stats = classification_epoch(
            model, loaders["train"], criterion, optimizer, scaler,
            max_batches=run_config.get("max_train_batches"), grad_clip=grad_clip,
            grad_accum=grad_accum, frozen_eval=frozen_eval,
        )
        val_stats = classification_epoch(
            model, loaders["val"], criterion,
            max_batches=run_config.get("max_eval_batches"),
        )
        if scheduler is not None:
            scheduler.step()
        row = {
            "epoch": epoch,
            "train_loss": train_stats["loss"], "train_accuracy": train_stats["accuracy"],
            "val_loss": val_stats["loss"], "val_accuracy": val_stats["accuracy"],
            "epoch_seconds": train_stats["seconds"],
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
        improved = (val_stats["accuracy"] > best_accuracy or
                    (val_stats["accuracy"] == best_accuracy and val_stats["loss"] < best_loss))
        if improved:
            best_accuracy, best_loss = val_stats["accuracy"], val_stats["loss"]
        state = {
            "model": model.state_dict(), "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "scaler": scaler.state_dict(), "epoch": epoch,
            "best_accuracy": best_accuracy, "best_loss": best_loss,
            "rng_state": rng_state(), "config": run_config,
        }
        atomic_torch_save(state, run_dir / "last.pt")
        if improved:
            atomic_torch_save(state, run_dir / "best.pt")
        print(f"epoch={epoch + 1}/{run_config['epochs']} train={train_stats['accuracy']:.4f} val={val_stats['accuracy']:.4f}")

    best = torch.load(run_dir / "best.pt", map_location=DEVICE, weights_only=False)
    model.load_state_dict(best["model"])
    val_stats = classification_epoch(model, loaders["val"], criterion,
                                     max_batches=run_config.get("max_eval_batches"))
    test_stats = classification_epoch(model, loaders["test"], criterion,
                                      max_batches=run_config.get("max_eval_batches"))
    metrics = {
        "run_mode": run_config["run_mode"], "best_epoch": best["epoch"],
        "val_loss": val_stats["loss"], "val_accuracy": val_stats["accuracy"],
        "test_loss": test_stats["loss"], "test_accuracy": test_stats["accuracy"],
        "peak_gpu_memory_mb": (torch.cuda.max_memory_allocated() / 2**20
                               if torch.cuda.is_available() else 0.0),
        **count_parameters(model),
    }
    y_true, y_pred, probabilities = predict_classifier(
        model, loaders["test"], max_batches=run_config.get("max_eval_batches"))
    np.savez_compressed(run_dir / "predictions.npz", y_true=y_true, y_pred=y_pred,
                        probabilities=probabilities)
    json_dump(metrics, run_dir / "metrics.json")
    return metrics, pd.DataFrame(history), (y_true, y_pred, probabilities)


@torch.no_grad()
def benchmark_model(model, sample, repeats=30, warmup=5):
    model.eval()
    sample = sample.to(DEVICE)
    for _ in range(warmup):
        model(sample)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    for _ in range(repeats):
        model(sample)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return {
        "latency_ms_per_image": 1000 * elapsed / (repeats * len(sample)),
        "benchmark_batch_size": len(sample),
        "benchmark_peak_memory_mb": (torch.cuda.max_memory_allocated() / 2**20
                                     if torch.cuda.is_available() else 0.0),
    }


def plot_history(history, path):
    frame = pd.DataFrame(history)
    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(frame.epoch, frame.train_loss, label="train")
    axes[0].plot(frame.epoch, frame.val_loss, label="validation")
    axes[0].set(title="Loss", xlabel="Epoch")
    axes[1].plot(frame.epoch, frame.train_accuracy, label="train")
    axes[1].plot(frame.epoch, frame.val_accuracy, label="validation")
    axes[1].set(title="Accuracy", xlabel="Epoch")
    for axis in axes:
        axis.legend(); axis.grid(alpha=.2)
    figure.tight_layout(); figure.savefig(path, dpi=160); plt.close(figure)


def plot_confusion(y_true, y_pred, class_names, path, normalize=True):
    matrix = confusion_matrix(y_true, y_pred, normalize="true" if normalize else None)
    size = max(7, min(20, len(class_names) * .22))
    figure, axis = plt.subplots(figsize=(size, size))
    sns.heatmap(matrix, cmap="Blues", xticklabels=class_names, yticklabels=class_names,
                annot=len(class_names) <= 10, fmt=".2f" if normalize else "d", ax=axis)
    axis.set(xlabel="Predicted", ylabel="True", title="Confusion matrix")
    figure.tight_layout(); figure.savefig(path, dpi=160); plt.close(figure)


def completed_metrics(run_dir, run_mode):
    path = Path(run_dir) / "metrics.json"
    if not path.exists():
        return None
    with path.open(encoding="utf-8") as handle:
        metrics = json.load(handle)
    return metrics if metrics.get("run_mode") == run_mode else None


seed_everything(CONFIG["seed"])
Path(CONFIG["data_root"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["output_root"]).mkdir(parents=True, exist_ok=True)


## 3. Dữ liệu và split dùng chung


### 3.1. Chuẩn bị Fashion-MNIST

**Mục tiêu:** Tạo lại đúng split 54.000/6.000/10.000 và normalization đã dùng trong E1/E2.

**Kết quả:** Tạo dataset, DataLoader, tên lớp và `SPLIT_HASH` để kiểm tra provenance của baseline.


In [ ]:
raw_train = datasets.FashionMNIST(CONFIG["data_root"], train=True, download=True)
mean = (raw_train.data.float() / 255.0).mean().item()
std = (raw_train.data.float() / 255.0).std().item()
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((mean,), (std,))])
train_full = datasets.FashionMNIST(CONFIG["data_root"], train=True, transform=transform, download=True)
test_dataset = datasets.FashionMNIST(CONFIG["data_root"], train=False, transform=transform, download=True)
train_idx, val_idx, SPLIT_HASH = fashion_split(raw_train, CONFIG["data_root"], CONFIG["seed"])
train_dataset, val_dataset = Subset(train_full, train_idx), Subset(train_full, val_idx)

loaders = {
    "train": DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True,
                        num_workers=CONFIG["num_workers"], pin_memory=AMP_ENABLED),
    "val": DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False,
                      num_workers=CONFIG["num_workers"], pin_memory=AMP_ENABLED),
    "test": DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False,
                       num_workers=CONFIG["num_workers"], pin_memory=AMP_ENABLED),
}
CLASS_NAMES = list(raw_train.classes)
print({"mean": mean, "std": std, "split_hash": SPLIT_HASH,
       "sizes": [len(train_dataset), len(val_dataset), len(test_dataset)]})


## 4. Biểu diễn chuỗi và mô hình


### 4.1. Xây dựng LSTM/GRU và baseline

**Mục tiêu:** Biểu diễn ảnh theo 28 hàng hoặc 49 patch, cài đặt LSTM/GRU một chiều và định nghĩa lại MLP/CNN của E1, trong đó CNN dùng Global Average Pooling.

**Kết quả:** Tạo bốn experiment recurrent, kiểm tra shape chuỗi và xác nhận GRU có thể overfit một batch nhỏ.


In [ ]:
class SequenceClassifier(nn.Module):
    def __init__(self, recurrent, representation, hidden_size=128):
        super().__init__()
        self.representation = representation
        input_size = 28 if representation == "row" else 16
        recurrent_class = nn.LSTM if recurrent == "lstm" else nn.GRU
        self.recurrent = recurrent_class(input_size, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 10)
    def as_sequence(self, images):
        if self.representation == "row":
            return images.squeeze(1)
        return F.unfold(images, kernel_size=4, stride=4).transpose(1, 2)
    def forward(self, images):
        sequence = self.as_sequence(images)
        output, state = self.recurrent(sequence)
        hidden = state[0] if isinstance(state, tuple) else state
        return self.head(hidden[-1])


class MLPClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(),
                                     nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))
    def forward(self, images): return self.network(images)


class CNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                      nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(64, 10)
    def forward(self, images): return self.head(self.gap(self.features(images)).flatten(1))


assert sum(parameter.numel() for parameter in CNNClassifier().parameters()) == 19466
EXPERIMENT_IDS = [f"{recurrent}_{representation}" for recurrent in ("lstm", "gru")
                  for representation in ("row", "patch")]
selected = CONFIG["experiment_ids"] or EXPERIMENT_IDS
unknown = set(selected) - set(EXPERIMENT_IDS)
if unknown: raise ValueError(f"Unknown experiment IDs: {sorted(unknown)}")

for representation, expected in (("row", (2, 28, 28)), ("patch", (2, 49, 16))):
    model = SequenceClassifier("lstm", representation)
    assert model.as_sequence(torch.randn(2, 1, 28, 28)).shape == expected
print("Sequence shapes and IDs are valid:", selected)

probe = SequenceClassifier("gru", "row").to(DEVICE)
images, labels = next(iter(loaders["train"])); images, labels = images[:32].to(DEVICE), labels[:32].to(DEVICE)
probe_optimizer = torch.optim.Adam(probe.parameters(), lr=3e-3)
probe_initial = None
for _ in range(10 if CONFIG["run_mode"] == "smoke" else 30):
    probe_optimizer.zero_grad(set_to_none=True)
    probe_loss = F.cross_entropy(probe(images), labels)
    if probe_initial is None: probe_initial = probe_loss.item()
    probe_loss.backward(); nn.utils.clip_grad_norm_(probe.parameters(), 1.0); probe_optimizer.step()
assert torch.isfinite(probe_loss) and probe_loss.item() < probe_initial
print("E3 one-batch overfit:", probe_initial, "->", probe_loss.item())
del probe, probe_optimizer
if torch.cuda.is_available(): torch.cuda.empty_cache()


## 5. Nhập baseline E1 có kiểm tra provenance


### 5.1. Đọc kết quả MLP/CNN từ E1

**Mục tiêu:** Tìm artifact `E1/full`, kiểm tra split hash, seed, experiment ID và `run_mode` trước khi tái sử dụng metric.

**Kết quả:** Tạo `baseline_results` và `missing_baselines`; baseline thiếu sẽ được chạy lại ở cell kế tiếp.


In [ ]:
def load_e1_baseline(experiment_id):
    run_dir = Path(CONFIG["output_root"]) / "E1" / "full" / experiment_id
    config_path, metrics_path = run_dir / "config.json", run_dir / "metrics.json"
    if not (config_path.exists() and metrics_path.exists()):
        return None
    config = json.loads(config_path.read_text(encoding="utf-8"))
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    if config.get("split_hash") != SPLIT_HASH or config.get("seed") != CONFIG["seed"]:
        raise RuntimeError(f"E1 artifact mismatch: {run_dir}")
    if config.get("experiment_id") != experiment_id or metrics.get("run_mode") != "full":
        raise RuntimeError(f"Invalid E1 artifact: {run_dir}")
    return {"experiment_id": f"e1_{experiment_id}", "source": "E1/full", **metrics}


baseline_results = [result for name in ("mlp", "cnn") if (result := load_e1_baseline(name)) is not None]
missing_baselines = [name for name in ("mlp", "cnn") if not any(row["experiment_id"] == f"e1_{name}" for row in baseline_results)]
print("Imported baselines:", [row["experiment_id"] for row in baseline_results])
print("Missing baselines (the next cell can reproduce them):", missing_baselines)


## 6. Huấn luyện và đánh giá


### 6.1. Chạy baseline thiếu và bốn mô hình chuỗi

**Mục tiêu:** Tái tạo baseline khi cần, huấn luyện LSTM/GRU với gradient clipping và đánh giá trên cùng split.

**Kết quả:** Lưu checkpoint, metrics, curves, confusion matrix và `summary.csv` kết hợp E1–E3.

**Lưu ý:** Bảng báo cáo chính nên dùng baseline E1/full thay vì baseline smoke được tái tạo tại đây.


In [ ]:
summary = list(baseline_results)
OUTPUT_BASE = Path(CONFIG["output_root"]) / "E3" / CONFIG["run_mode"]

# In smoke mode, reproduce missing baselines to validate the E1 comparison path.
# For a report table, run E1/full first or change this notebook to full.
baseline_factories = {"mlp": MLPClassifier, "cnn": CNNClassifier}
for baseline_name in missing_baselines:
    run_dir = OUTPUT_BASE / f"baseline_{baseline_name}"
    seed_everything(CONFIG["seed"])
    model = baseline_factories[baseline_name]().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    baseline_config = {
        **CONFIG, "assignment": "E3", "experiment_id": f"baseline_{baseline_name}",
        "split_hash": SPLIT_HASH, "max_train_batches": 4 if CONFIG["run_mode"] == "smoke" else None,
        "max_eval_batches": 2 if CONFIG["run_mode"] == "smoke" else None,
    }
    metrics, history, predictions = fit_classifier(model, loaders, optimizer, None, run_dir, baseline_config)
    plot_history(history, run_dir / "curves.png")
    summary.append({"experiment_id": f"baseline_{baseline_name}", "source": "reproduced", **metrics})

for experiment_id in selected:
    recurrent, representation = experiment_id.split("_", 1)
    run_dir = OUTPUT_BASE / experiment_id
    existing = completed_metrics(run_dir, CONFIG["run_mode"])
    if existing is not None and not CONFIG["evaluate_only"]:
        summary.append({"experiment_id": experiment_id, "source": "E3", **existing}); continue
    seed_everything(CONFIG["seed"])
    model = SequenceClassifier(recurrent, representation).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    run_config = {
        **CONFIG, "assignment": "E3", "experiment_id": experiment_id,
        "recurrent": recurrent, "representation": representation, "split_hash": SPLIT_HASH,
        "sequence_length": 28 if representation == "row" else 49,
        "input_size": 28 if representation == "row" else 16,
        "max_train_batches": 4 if CONFIG["run_mode"] == "smoke" else None,
        "max_eval_batches": 2 if CONFIG["run_mode"] == "smoke" else None,
    }
    metrics, history, predictions = fit_classifier(
        model, loaders, optimizer, None, run_dir, run_config, grad_clip=1.0)
    sample_images, _ = next(iter(loaders["test"]))
    metrics.update(benchmark_model(model, sample_images[:32], repeats=5 if CONFIG["run_mode"] == "smoke" else 30))
    json_dump(metrics, run_dir / "metrics.json")
    plot_history(history, run_dir / "curves.png")
    plot_confusion(predictions[0], predictions[1], CLASS_NAMES, run_dir / "confusion_matrix.png")
    summary.append({"experiment_id": experiment_id, "source": "E3", **metrics})

summary_frame = pd.DataFrame(summary).sort_values("experiment_id")
summary_frame.to_csv(OUTPUT_BASE / "summary.csv", index=False)
display(summary_frame)


## 7. Câu hỏi phân tích

1. Row và patch representation thay đổi độ dài chuỗi, input dimension và latency ra sao?
2. LSTM/GRU nào đạt trade-off tốt hơn giữa accuracy và số tham số?
3. Thứ tự quét raster tạo giả định nào về quan hệ không gian?
4. Những lỗi nào CNN xử lý tốt hơn recurrent model, và quan sát này có phù hợp với inductive bias của convolution không?
